# Busan Traffic AI PBL - 15_busan_ml_model_training.ipynb

부산시 교통량 예측 프로젝트 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
# =========================
# [셀 1] 라이브러리 로드
# =========================
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# =========================
# [셀 2] 데이터 로드
# =========================
X_path = "data/processed/merged_x.csv"
y_path = "data/processed/traffic_daily_total_2022_2023.csv"

df_x = pd.read_csv(X_path)
df_y = pd.read_csv(y_path)

print(df_x.shape, df_y.shape)
# df_x.head()
df_y.head()

In [ ]:
# =========================
# [셀 3] date 처리 및 y 컬럼 자동 탐지
# =========================
df_x["date"] = pd.to_datetime(df_x["date"])
df_y["date"] = pd.to_datetime(df_y["date"])

# y 후보: date 제외한 컬럼들 중 하나를 선택
y_candidates = [c for c in df_y.columns if c != "date"]
if len(y_candidates) == 1:
    y_col = y_candidates[0]
else:
    # 흔한 이름 우선
    preferred = ["y", "target", "traffic", "traffic_total", "daily_total", "label"]
    found = [c for c in preferred if c in y_candidates]
    y_col = found[0] if len(found) > 0 else y_candidates[-1]

print("사용할 y 컬럼:", y_col)
df_y[[ "date", y_col ]].head()

In [ ]:
# =========================
# [셀 4] X와 y 병합
# =========================
df = df_x.merge(df_y[["date", y_col]], on="date", how="inner").sort_values("date").reset_index(drop=True)
print(df.shape)
df.head()

In [ ]:
# =========================
# [셀 5] Train Val Test Split
# =========================
train_mask = (df["date"] >= "2022-01-01") & (df["date"] <= "2022-12-31")
val_mask   = (df["date"] >= "2023-01-01") & (df["date"] <= "2023-01-31")
test_mask  = (df["date"] >= "2023-02-01") & (df["date"] <= "2023-12-31")

df_train = df[train_mask].copy()
df_val   = df[val_mask].copy()
df_test  = df[test_mask].copy()

print("train/val/test:", df_train.shape, df_val.shape, df_test.shape)

In [ ]:
# =========================
# [셀 6] Feature/Label 분리
# =========================
feature_cols = [c for c in df.columns if c not in ["date", y_col]]

X_train, y_train = df_train[feature_cols], df_train[y_col]
X_val, y_val     = df_val[feature_cols], df_val[y_col]
X_test, y_test   = df_test[feature_cols], df_test[y_col]

print(len(feature_cols), "features")

In [ ]:
# =========================
# [셀 7] 전처리 파이프라인
# - 수치형: 결측 median 대치 + 표준화
# - 문자열/범주형: (있다면) 결측 most_frequent 대치
# =========================
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_train.columns if c not in num_cols]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ],
    remainder="drop"
)

print("num:", len(num_cols), "cat:", len(cat_cols))

In [ ]:
# =========================
# [셀 8] 모델 정의
# =========================
models = {
    "kNN": KNeighborsRegressor(n_neighbors=7),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
}

In [ ]:
# =========================
# [셀 9] 학습 및 평가 함수
# =========================
def evaluate_regression(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)

    val_pred = pipe.predict(X_val)
    test_pred = pipe.predict(X_test)

    r_val = evaluate_regression(y_val, val_pred)
    r_test = evaluate_regression(y_test, test_pred)

    results.append({
        "model": name,
        "val_MAE": r_val["MAE"],
        "val_RMSE": r_val["RMSE"],
        "val_R2": r_val["R2"],
        "test_MAE": r_test["MAE"],
        "test_RMSE": r_test["RMSE"],
        "test_R2": r_test["R2"],
    })

pd.DataFrame(results).sort_values("val_RMSE")